# M12 Lab — Local LLM Mastery

**Datasets:** your own notebooks + CSVs &nbsp;|&nbsp; **Focus:** Ollama, synthetic data, RAG, privacy-aware local AI


In [ ]:
MODULE_ID = "M12"
# ── Confusion reporter ───────────────────────────────────────────────────────
# Run this cell once to set up the reporter, then call it any time you are
# confused about a term or concept. It logs the entry to the instructor
# dashboard so they can address common pain-points.
#
#   Usage (in any later cell):
#       await im_confused("overfitting")
#       await im_confused("gradient descent", "not sure how the learning rate affects convergence")

import json as _json

async def im_confused(term: str, note: str = ""):
    """Report a confusing term to your instructor.
    
    Args:
        term: the word / concept that confused you (e.g. "overfitting")
        note: optional extra detail (e.g. "what does the bias-variance tradeoff mean here?")
    """
    try:
        from pyodide.http import pyfetch
        payload = {
            "word": term,
            "message": note,
            "source": "self_report",
            "moduleId": MODULE_ID,
        }
        resp = await pyfetch(
            "/api/error-log",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps(payload),
            credentials="include",
        )
        if resp.ok:
            print(f"✅ Reported \"{term}\" — your instructor will see this in the Confusion dashboard.")
        else:
            print(f"⚠️  Could not report (HTTP {resp.status}). Are you logged in to DataPath?")
    except ImportError:
        # Running outside JupyterLite (e.g. plain Jupyter / Docker)
        import requests as _req
        payload = {"word": term, "message": note, "source": "self_report", "moduleId": MODULE_ID}
        try:
            r = _req.post("http://localhost:3001/api/error-log", json=payload, timeout=5)
            print("✅ Reported!" if r.ok else f"⚠️  HTTP {r.status_code}")
        except Exception as e:
            print(f"⚠️  Could not reach DataPath server: {e}")
    except Exception as e:
        print(f"⚠️  Unexpected error: {e}")

print("✓ Confusion reporter ready.")
print("  Usage: await im_confused(\"term you found confusing\")")


In [ ]:
import requests

r = requests.get("http://localhost:11434/api/tags", timeout=10)
print("status:", r.status_code)
print(r.json())


## L12.2 — Basic Ollama API call


In [ ]:
import json as _json

payload_messages = [{"role": "user", "content": "Explain in 2 sentences why local LLMs matter for data science."}]

try:
    from pyodide.http import pyfetch
    resp = await pyfetch(
        "http://localhost:11434/api/chat",
        method="POST",
        headers={"Content-Type": "application/json"},
        body=_json.dumps({"model": "gemma4n", "messages": payload_messages, "stream": False}),
    )
    data = await resp.json()
    print(data["message"]["content"])
except ImportError:
    import requests
    r = requests.post("http://localhost:11434/api/chat",
                      json={"model": "gemma4n", "messages": payload_messages, "stream": False}, timeout=60)
    r.raise_for_status()
    print(r.json()["message"]["content"])


## L12.2 — Streaming output


In [ ]:
import json as _json

messages = [{"role": "user", "content": "List 3 privacy benefits of a local LLM for students."}]

# Note: JupyterLite can't do true streaming — we use stream:False and print all at once.
try:
    from pyodide.http import pyfetch
    print("⏳ Generating… (JupyterLite buffers the full response)")
    resp = await pyfetch(
        "http://localhost:11434/api/chat",
        method="POST",
        headers={"Content-Type": "application/json"},
        body=_json.dumps({"model": "gemma4n", "messages": messages, "stream": False}),
    )
    data = await resp.json()
    print(data["message"]["content"])
except ImportError:
    import requests
    with requests.post("http://localhost:11434/api/chat",
                       json={"model": "gemma4n", "messages": messages, "stream": True},
                       stream=True, timeout=60) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line: continue
            chunk = _json.loads(line.decode("utf-8"))
            if "message" in chunk and "content" in chunk["message"]:
                print(chunk["message"]["content"], end="", flush=True)
    print()


## L12.3 — Structured JSON output


In [ ]:
import json as _json

prompt = """
Return ONLY a valid JSON array with 3 rows.
Each row must contain: hypothesis, metric, caution.
Context: churn dataset with columns tenure, monthly_spend, churn.
"""

try:
    from pyodide.http import pyfetch
    resp = await pyfetch(
        "http://localhost:11434/api/chat",
        method="POST",
        headers={"Content-Type": "application/json"},
        body=_json.dumps({"model": "gemma4n", "messages": [{"role": "user", "content": prompt}], "stream": False}),
    )
    data = await resp.json()
    raw = data["message"]["content"]
except ImportError:
    import requests
    r = requests.post("http://localhost:11434/api/chat",
                      json={"model": "gemma4n", "messages": [{"role": "user", "content": prompt}], "stream": False},
                      timeout=60)
    r.raise_for_status()
    raw = r.json()["message"]["content"]

# Parse JSON from response
try:
    start = raw.find("[")
    end   = raw.rfind("]") + 1
    hypotheses = _json.loads(raw[start:end])
    for h in hypotheses:
        print(h)
except _json.JSONDecodeError:
    print("⚠️  Model returned invalid JSON. Raw response:")
    print(raw)


## L12.4 — Synthetic data generation [🔮 ai-generated]


In [ ]:
import json as _json, pandas as pd

async def ask_gemma(prompt: str, model: str = "gemma4n") -> str:
    """Send a prompt to Ollama. Works in JupyterLite (pyfetch) and Docker (requests)."""
    try:
        from pyodide.http import pyfetch
        resp = await pyfetch(
            "http://localhost:11434/api/chat",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps({"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False}),
        )
        data = await resp.json()
    except ImportError:
        import requests
        r = requests.post("http://localhost:11434/api/chat",
                          json={"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False},
                          timeout=120)
        r.raise_for_status()
        data = r.json()
    return data["message"]["content"]

schema_prompt = """
Generate 20 rows of realistic e-commerce order data as a JSON array.
Each row: {\"order_id\": int, \"customer_age\": int, \"product_category\": str, \"order_value\": float, \"returned\": bool}
Return ONLY the JSON array, no other text.
"""

raw = await ask_gemma(schema_prompt)
try:
    start = raw.find('[')
    end = raw.rfind(']') + 1
    rows = _json.loads(raw[start:end])
    synth = pd.DataFrame(rows)
    print(synth.head())
except Exception as e:
    print('Parse error:', e)
    print(raw[:300])


## L12.4 — Validate synthetic data [AI-VERIFY]


In [ ]:
import pandera as pa
from pandera import Check, Column, DataFrameSchema

schema = DataFrameSchema(
    {
        "order_id": Column(int),
        "customer_age": Column(int, checks=Check.in_range(18, 75)),
        "product_category": Column(str),
        "order_value": Column(float, checks=Check.in_range(5.0, 500.0)),
        "days_to_delivery": Column(int, checks=Check.in_range(1, 14)),
        "returned": Column(bool),
    }
)

validated_df = schema.validate(df)
print(validated_df.head())


## L12.5 — RAG pipeline


In [ ]:
        import importlib.util
        import json
        from pathlib import Path

        missing = [
            pkg for pkg in ["sentence_transformers", "faiss", "numpy", "pandas"]
            if importlib.util.find_spec(pkg) is None
        ]
        if missing:
            raise ImportError(f"Missing packages for this cell: {missing}")

        import faiss
        import numpy as np
        import pandas as pd
        import requests
        from sentence_transformers import SentenceTransformer

        def load_documents(folder: str = "."):
            docs = []
            for path in Path(folder).glob("*.ipynb"):
                nb = json.loads(path.read_text(encoding="utf-8"))
                docs.append({"source": path.name, "text": "
".join("".join(c.get("source", [])) for c in nb.get("cells", []))})
            for path in Path(folder).glob("*.csv"):
                frame = pd.read_csv(path)
                docs.append({
                    "source": path.name,
                    "text": f"Columns: {list(frame.columns)}
Shape: {frame.shape}
Preview:
{frame.head(5).to_csv(index=False)}",
                })
            return docs

        def chunk_text(text: str, size: int = 400, overlap: int = 80):
            chunks = []
            start = 0
            while start < len(text):
                chunks.append(text[start:start + size])
                start += size - overlap
            return chunks

        docs = load_documents(".")
        chunk_records = []
        for doc in docs:
            for chunk in chunk_text(doc["text"]):
                chunk_records.append({"source": doc["source"], "chunk": chunk})

        embedder = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = np.asarray(embedder.encode([r["chunk"] for r in chunk_records], normalize_embeddings=True), dtype="float32")
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

        query = "Which local files mention confidence intervals, p-values, or effect size?"
        query_vec = np.asarray(embedder.encode([query], normalize_embeddings=True), dtype="float32")
        _, idxs = index.search(query_vec, k=min(3, len(chunk_records)))
        retrieved = [chunk_records[i] for i in idxs[0]]
        context = "

".join(f"[{r['source']}]
{r['chunk']}" for r in retrieved)

        answer = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "gemma4n",
                "messages": [{
                    "role": "user",
                    "content": f"Use only the retrieved context below. If unsupported, say so.

Context:
{context}

Question: {query}",
                }],
                "stream": False,
            },
            timeout=120,
        )
        answer.raise_for_status()
        print("retrieved sources:", [r["source"] for r in retrieved])
        print(answer.json()["message"]["content"])


## L12.7 [AI-OFF] — Privacy design exercise


In a new markdown cell below this one, design a privacy-preserving local AI workflow for a data-science project.

Your answer must classify each stage as **AI-assisted**, **AI-generated**, or **AI-independent**.
Include at least these stages:
1. loading data,
2. retrieving notebook context,
3. generating a chart narrative,
4. verifying the final result,
5. logging the interaction in `AI_USE.md`.

Then identify at least 3 leakage risks and 3 mitigations.
Do this **without** model assistance.
